# Explainability Evaluation - Reconstruction Error Agreement

## Testing the Attention based Explainability and SHAP on Agreement with Reconstruction Error

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# Connect and Access Git Repository

import os
import sys
import json
from getpass import getpass
from pathlib import Path

GITHUB_USERNAME = "sharaffatima"
REPO_NAME = "bachelor-thesis"

token = getpass("GitHub token: ")

REPO_URL = f"https://{token}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
REPO_DIR = Path("/content") / REPO_NAME


if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
sys.path.append(str(REPO_DIR))

print("Repo exists:", REPO_DIR.exists())

GitHub token: ··········
Cloning into '/content/bachelor-thesis'...
remote: Enumerating objects: 687, done.
remote: Counting objects: 100% (202/202), done.
remote: Compressing objects: 100% (142/142), done.
remote: Total 687 (delta 125), reused 110 (delta 59), pack-reused 485 (from 2)
Receiving objects: 100% (687/687), 36.27 MiB | 21.63 MiB/s, done.
Resolving deltas: 100% (365/365), done.
/content/bachelor-thesis
Repo exists: True


In [4]:
!git pull

Already up to date.


In [ ]:
import os
import numpy as np
import pandas as pd
import shap
import torch
from torch.utils.data import DataLoader, TensorDataset

# global configuration

REPO_DIR = Path("/content/bachelor-thesis")
sys.path.append(str(REPO_DIR))
sys.path.append(str(REPO_DIR / "explainability"))


from utils.dataset import PTBXLDataset
from utils.seed import set_seed, RANDOM_SEED
from explainability.anoformer_attention import AnoFormerExplainability
from explainability.tranad_attention import TranADExplainability
from models.anoformer import AnoFormerMainModel
from models.tranad import TranADMainModel
from explainability.evaluation import ReconstructionErrorAgreementEvaluation
from explainability.sample_selection import load_samples_from_dataset


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(RANDOM_SEED)
print(f"Active device: {DEVICE}")

Active device: cuda


In [ ]:
# Paths to Directories
# Load processed data and pretrained models

DATA_DIR = Path("/content/drive/MyDrive/ECG data")
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = DATA_DIR / "results"
SHAP_RESULTS_DIR = RESULTS_DIR / "shap"

MODEL_DIR = RESULTS_DIR / "models"
ANOFORMER_MODEL_PATH = MODEL_DIR / "anoformer" / "anoformer_40epochs_larger_capacity_2.pt"
TRANAD_MODEL_PATH = MODEL_DIR / "tranad" / "tranad_20epochs_larger_capacity.pt"

AGREEMENT_DIR = RESULTS_DIR / "evaluation" / "reconstruction_error_agreement"
AGREEMENT_DIR .mkdir(parents=True, exist_ok=True)

In [8]:
# Load my preprocessed training and test datasets

X_train = np.load(PROCESSED_DIR / "X_train.npy")
y_train = np.load(PROCESSED_DIR / "y_train.npy")

X_test = np.load(os.path.join(PROCESSED_DIR, "X_test.npy"))
y_test = np.load(os.path.join(PROCESSED_DIR, "y_test.npy"))

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(17418, 1000, 12) (17418,)
(2198, 1000, 12) (2198,)


In [9]:
# select only normal ecgs from Training set
normal_train_mask = y_train == 0
X_train_normal = X_train[normal_train_mask]
y_train_normal = y_train[normal_train_mask]

# Datasets
# We need the Train dataset because the SHAP background is from the training set. We also need the mean and std of the training set to normalize the test set.
train_dataset = PTBXLDataset(
    X_train_normal,
    y_train_normal,
    normalize=True
)

# We need the Test dataset because the chosen samples for evaluation are from the test set
test_dataset = PTBXLDataset(
    X_test,
    y_test,
    mean=train_dataset.mean,
    std=train_dataset.std,
    normalize=True
)

# Dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    drop_last=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False
)

In [ ]:
# Load fixed evaluation samples

EvaluationSamplePath = (
    RESULTS_DIR
    / "evaluation"
    / "evaluation_samples.npz"
)

EvaluationSamples = load_samples_from_dataset(
    sample_path=EvaluationSamplePath,
    test_dataset=test_dataset,
    device=None,
)

print("Number of evaluation samples:", len(EvaluationSamples))

for sample in EvaluationSamples:
    print(
        sample["index"],
        sample["case_type"],
        sample["label"],
        sample["ecg"].shape,
    )

Index of Normal ECG: 583 Label: 0
Index of Anomalous ECG: 495 Label: 1
Normal ECG shape: torch.Size([1, 1000, 12])
Anomalous ECG shape: torch.Size([1, 1000, 12])


In [11]:
# Create the Evaluator instance

AgreementEvaluator = ReconstructionErrorAgreementEvaluation()

# AnoFormer

In [12]:
# Create AnoFormer Model

AnoFormer = AnoFormerMainModel(
    NumFeatures=12,
    ModelDimension=64,
    NumHeads=4,
    NumEncoderLayers=2,
    NumDecoderLayers=2,
    DiscriminatorLayers=1,
    ForwardDimension=128,
    DropoutRate=0.1,
    GeneratorLearningRate=1e-4,
    DiscriminatorLearningRate=1e-4,
    MaskingRate=0.15,
    AnomalyThreshold=0.75,
    ReconstructionWeight=10.0,
)

# Load the pretrained model
AnoFormer.Load(str(ANOFORMER_MODEL_PATH))

# Create Explainer to extract Attention Values
AnoFormerExplainer = AnoFormerExplainability(AnoFormer)


Model loaded from: /content/drive/MyDrive/ECG data/results/models/anoformer/anoformer_40epochs_larger_capacity_2.pt


In [ ]:
# Extract the reconstruction Error for the normal and anomalous ECGs
AnoFormerReconstructionErrorValues = []

for sample in EvaluationSamples:
    ecg = sample["ecg"].to(AnoFormer.Device)

    _, _, reconstruction_error = (
        AnoFormerExplainer.get_reconstruction_error_over_time(
            TimeSeriesWindow=ecg,
            DatasetForInverseNorm=test_dataset,
        )
    )

    AnoFormerReconstructionErrorValues.append(
        reconstruction_error
    )

    print(
        "AnoFormer reconstruction error:",
        sample["index"],
        sample["case_type"],
        reconstruction_error.shape,
    )

### Attention Evaluation

In [ ]:
# Extract the attention importance values for the normal and anomalous ECGs

AnoFormerAttentionImportanceValues = []

for sample in EvaluationSamples:
    ecg = sample["ecg"].to(AnoFormer.Device)

    attention_importance = AnoFormerExplainer.get_attention_importance(
        TimeSeriesWindow=ecg,
        BlockIndex=0,
        HeadIndex=None,
    )

    AnoFormerAttentionImportanceValues.append(
        attention_importance
    )

    print(
        "AnoFormer attention:",
        sample["index"],
        sample["case_type"],
        attention_importance.shape,
    )

Normal attention importance: (1000,)
Anomalous attention importance: (1000,)


In [ ]:
# Evaluate

AnoFormerAttentionAgreementList = (
    AgreementEvaluator.evaluate_samples(
        samples=EvaluationSamples,
        temporal_importance_values=AnoFormerAttentionImportanceValues,
        reconstruction_error_values=AnoFormerReconstructionErrorValues,
        model_name="AnoFormer",
        explanation_name="Attention",
    )
)

AnoFormerAttentionAgreementResults = pd.DataFrame(
    AnoFormerAttentionAgreementList
)

AnoFormerAttentionAgreementResults

In [ ]:
AnoFormerAttentionAgreementSummary = (
    AnoFormerAttentionAgreementResults
    .groupby(
        [
            "Model",
            "Explanation",
            "Case Type",
        ]
    )["Spearman Correlation"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

AnoFormerAttentionAgreementSummary

In [ ]:
AnoFormerAttentionAgreementResults.to_csv(
    AGREEMENT_DIR / "anoformer_attention_reconstruction_error_agreement_results.csv",
    index=False,
)

AnoFormerAttentionAgreementSummary.to_csv(
    AGREEMENT_DIR / "anoformer_attention_reconstruction_error_agreement_summary.csv",
    index=False,
)

## SHAP Evaluation

In [ ]:
# Load shap values

AnoFormerSHAPValues = np.load(
    SHAP_RESULTS_DIR
    / "anoformer"
    / "anoformer_evaluation_shap_values.npy"
)

print("AnoFormer SHAP values:", AnoFormerSHAPValues.shape)

In [ ]:
# transform SHAP values temporal importance

AnoFormerSHAPImportanceValues = []

for shap_values in AnoFormerSHAPValues:
    temporal_importance = np.abs(
        shap_values
    ).sum(axis=1)

    AnoFormerSHAPImportanceValues.append(
        temporal_importance
    )

print(
    "Number of AnoFormer SHAP importance arrays:",
    len(AnoFormerSHAPImportanceValues),
)

print(
    "Shape of first AnoFormer SHAP importance:",
    AnoFormerSHAPImportanceValues[0].shape,
)

In [ ]:
# Evaluate

AnoFormerSHAPAgreementList = (
    AgreementEvaluator.evaluate_samples(
        samples=EvaluationSamples,
        temporal_importance_values=AnoFormerSHAPImportanceValues,
        reconstruction_error_values=AnoFormerReconstructionErrorValues,
        model_name="AnoFormer",
        explanation_name="SHAP",
    )
)

AnoFormerSHAPAgreementResults = pd.DataFrame(
    AnoFormerSHAPAgreementList
)

AnoFormerSHAPAgreementResults

In [ ]:
AnoFormerSHAPAgreementSummary = (
    AnoFormerSHAPAgreementResults
    .groupby(
        [
            "Model",
            "Explanation",
            "Case Type",
        ]
    )["Spearman Correlation"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

AnoFormerSHAPAgreementSummary

In [ ]:
# Save the Results

AnoFormerSHAPAgreementResults.to_csv(
    AGREEMENT_DIR / "anoformer_shap_reconstruction_error_agreement_results.csv",
    index=False,
)

AnoFormerSHAPAgreementSummary.to_csv(
    AGREEMENT_DIR / "anoformer_shap_reconstruction_error_agreement_summary.csv",
    index=False,
)

# TranAD

In [ ]:
# Create TranAD Model

TranAD = TranADMainModel(
    NumFeatures=12,
    ModelDimension=64,
    NumHeads=4,
    NumEncoderLayers=2,
    NumDecoderLayers=2,
    ForwardDimension=128,
    DropoutRate=0.1,
    LearningRate=1e-4,
    FocusGrowthRate=1.05,
    InitialFocusScore=1.0,
    MaxFocusScore=5.0,
    AnomalyPercentile=95.0,
)

# Load the pretrained model
TranAD.Load(str(TRANAD_MODEL_PATH))

# Create Explainer to extract Attention Values
TranADExplainer = TranADExplainability(TranAD)

In [ ]:
# Extract the reconstruction Error for the normal and anomalous ECGs

TranADReconstructionErrorValues = []

for sample in EvaluationSamples:
    ecg = sample["ecg"].to(TranAD.Device)

    _, _, reconstruction_error = (
        TranADExplainer.get_reconstruction_error_over_time(
            TimeSeriesWindow=ecg,
            DatasetForInverseNorm=test_dataset,
        )
    )

    TranADReconstructionErrorValues.append(
        reconstruction_error
    )

    print(
        "TranAD reconstruction error:",
        sample["index"],
        sample["case_type"],
        reconstruction_error.shape,
    )


### Attention Evaluation

In [ ]:
# Extract Attention Values of Original ECGs

TranADAttentionImportanceValues = []

for sample in EvaluationSamples:
    ecg = sample["ecg"].to(TranAD.Device)

    attention_importance = TranADExplainer.get_attention_importance(
        TimeSeriesWindow=ecg,
        LayerIndex=0,
        HeadIndex=None,
    )

    TranADAttentionImportanceValues.append(
        attention_importance
    )

    print(
        "TranAD attention:",
        sample["index"],
        sample["case_type"],
        attention_importance.shape,
    )

In [ ]:
# Evaluate

TranADAttentionAgreementList = (
    AgreementEvaluator.evaluate_samples(
        samples=EvaluationSamples,
        temporal_importance_values=TranADAttentionImportanceValues,
        reconstruction_error_values=TranADReconstructionErrorValues,
        model_name="TranAD",
        explanation_name="Attention",
    )
)

TranADAttentionAgreementResults = pd.DataFrame(
    TranADAttentionAgreementList
)

TranADAttentionAgreementResults

In [ ]:
TranADAttentionAgreementSummary = (
    TranADAttentionAgreementResults
    .groupby(
        [
            "Model",
            "Explanation",
            "Case Type",
        ]
    )["Spearman Correlation"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

TranADAttentionAgreementSummary

In [ ]:
# Save the Results

TranADAttentionAgreementResults.to_csv(
    AGREEMENT_DIR / "tranad_attention_reconstruction_error_agreement_results.csv",
    index=False,
)

TranADAttentionAgreementSummary.to_csv(
    AGREEMENT_DIR / "tranad_attention_reconstruction_error_agreement_summary.csv",
    index=False,
)

## SHAP Evaluation

In [ ]:
# Load shap values

TranADSHAPValues = np.load(
    SHAP_RESULTS_DIR
    / "tranad"
    / "tranad_evaluation_shap_values.npy"
)

print("TranAD SHAP values:", TranADSHAPValues.shape)

In [ ]:
# transform SHAP values into temporal importance

TranADSHAPImportanceValues = []

for shap_values in TranADSHAPValues:
    temporal_importance = np.abs(
        shap_values
    ).sum(axis=1)

    TranADSHAPImportanceValues.append(
        temporal_importance
    )

print(
    "Number of TranAD SHAP importance arrays:",
    len(TranADSHAPImportanceValues),
)

print(
    "Shape of first TranAD SHAP importance:",
    TranADSHAPImportanceValues[0].shape,
)

In [ ]:
# Evaluate

TranADSHAPAgreementList = (
    AgreementEvaluator.evaluate_samples(
        samples=EvaluationSamples,
        temporal_importance_values=TranADSHAPImportanceValues,
        reconstruction_error_values=TranADReconstructionErrorValues,
        model_name="TranAD",
        explanation_name="SHAP",
    )
)

TranADSHAPAgreementResults = pd.DataFrame(
    TranADSHAPAgreementList
)

TranADSHAPAgreementResults

In [ ]:
TranADSHAPAgreementSummary = (
    TranADSHAPAgreementResults
    .groupby(
        [
            "Model",
            "Explanation",
            "Case Type",
        ]
    )["Spearman Correlation"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

TranADSHAPAgreementSummary

In [ ]:
# Save the Results

TranADSHAPAgreementResults.to_csv(
    AGREEMENT_DIR / "tranad_shap_reconstruction_error_agreement_results.csv",
    index=False,
)

TranADSHAPAgreementSummary.to_csv(
    AGREEMENT_DIR / "tranad_shap_reconstruction_error_agreement_summary.csv",
    index=False,
)

Saved to: /content/drive/MyDrive/ECG data/results/evaluation/reconstruction_error_agreement/tranad_shap_reconstruction_error_agreement_results.csv


# Final Reconstruction Error Agreement Evaluation Results

In [ ]:
AgreementSummary = pd.concat(
    [
        AnoFormerAttentionAgreementSummary,
        AnoFormerSHAPAgreementSummary,
        TranADAttentionAgreementSummary,
        TranADSHAPAgreementSummary,
    ],
    ignore_index=True,
)

AgreementSummary

In [ ]:
AgreementSummary.to_csv(
    AGREEMENT_DIR / "reconstruction_error_agreement_summary_all_methods.csv",
    index=False,
)